In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
# from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
import tqdm
import random
import os
import json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
# load MNIST data
transform = transforms.ToTensor()
train_data  = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)


100%|██████████| 9.91M/9.91M [00:00<00:00, 41.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.05MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.01MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.66MB/s]


In [4]:
# firstly see how the data look like and size of dataset
img, label = train_data[16]
print(img.size)
# plt.imshow(img)
print(f"label is {label}, and image size is {img.size}\n")
print(f"train data size {len(train_data)}, test data size {len(test_data)}")

<built-in method size of Tensor object at 0x78b60970fed0>
label is 2, and image size is <built-in method size of Tensor object at 0x78b60970fed0>

train data size 60000, test data size 10000


In [5]:
# process data get a train with 20 points, valid with 100 points rest of train are pooling points
# train and valid need random and balanced
len_train = len(train_data)
index_lst = []
for i in range(10):
  data_index = [index for index in range(len_train) if train_data[index][1] == i ]
  index_lst.append(data_index)

# train_index = []
# valid_index = []
# for i in range(10):
#   temp_choose = random.sample(index_lst[i], k=12)
#   train_index.extend(temp_choose[:2])
#   valid_index.extend(temp_choose[2:])
#   index_lst[i] = [x for x in index_lst[i] if x not in temp_choose]
# pooling_index = [i for j in  index_lst for i in j]
# len(pooling_index)

In [6]:
train_index = []
valid_index = []
for i in range(10):
  temp_choose = random.sample(index_lst[i], k=12)
  train_index.extend(temp_choose[:2])
  valid_index.extend(temp_choose[2:])
  index_lst[i] = [x for x in index_lst[i] if x not in temp_choose]
pooling_index = [i for j in  index_lst for i in j]
len(pooling_index)

59880

In [7]:
pooling_index = random.sample(pooling_index, k=3000)
temp = pooling_index.copy()
pooling_index.extend(temp)
pooling_index.extend(temp)
random.shuffle(pooling_index)
print(len(pooling_index))
print(len(set(pooling_index)))

9000
3000


In [8]:
# path = '/content/drive/MyDrive/UDL_DATA/'
# train_file_name = 'train_indices_extend.json'
# pool_file_name = 'pooling_indices_new.json'
# train_file_path = os.path.join(path, train_file_name)
# pool_file_path = os.path.join(path, pool_file_name)
# # with open(train_file_path, 'w') as f:
# #     json.dump(train_index, f)
# with open(train_file_path, 'r') as f:
#     train_index = json.load(f)
# with open(pool_file_path, 'r') as f:
#     pooling_index = json.load(f)
# print(len(pooling_index))

In [9]:
class NewDataset(Dataset):
  def __init__(self, index):
    imgs = []
    labels = []
    for i in index:
      img, label = train_data[i]
      imgs.append(img)
      labels.append(label)
    self.imgs = torch.stack(imgs)
    self.labels = torch.tensor(labels)

  def __len__(self):
    return len(self.imgs)

  def __getitem__(self, idx):
    return self.imgs[idx], self.labels[idx]



In [10]:
class BaseCNN(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.covn1 =nn.Conv2d(in_channels=1, out_channels=32,  kernel_size=4)
    self.covn2 = nn.Conv2d(in_channels=32, out_channels=32,  kernel_size=4)
    self.max_pool = nn.MaxPool2d(kernel_size=2)
    self.dropout_layer1 = nn.Dropout(p=0.25)
    self.flatten = nn.Flatten()
    self.dense_layer1 = nn.Linear(in_features=3872, out_features=128)
    self.dropout_layer2 = nn.Dropout(p=0.5)
    self.dense_layer2 = nn.Linear(in_features=128, out_features=10)

  def forward(self, x):
    x = F.relu(self.covn1(x))
    x = F.relu(self.covn2(x))
    x = self.max_pool(x)
    x = self.dropout_layer1(x)
    x = self.flatten(x)
    x = F.relu(self.dense_layer1(x))
    x = self.dropout_layer2(x)
    # print(x.shape)
    x = self.dense_layer2(x)
    # x = F.softmax(x, dim=1)
    return x

In [11]:
# write train function
def train_model(trainData):
    train_loader  = DataLoader(trainData, batch_size=128, shuffle=True)
    model = BaseCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    weight_decay = 0.02/(len(trainData))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    # Training loop
    for epoch in tqdm.tqdm(range(50)):
      total_loss = 0
      for i, (imgs, labels) in (enumerate(train_loader)):
        optimizer.zero_grad()
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
      avg_loss = total_loss / len(train_loader)
      # if epoch % 79 == 0:
      #   print(f"/n epoch{epoch}, train_loss is {avg_loss:.4f}.")

    return model



In [12]:
def random_choose(pooling_index, acq_size=10):
  new_data_index = random.sample(pooling_index, k=acq_size)
  new_pooling_index = [i for i in pooling_index if i not in new_data_index]
  return new_data_index, new_pooling_index



In [13]:
# calculate test accuracy
def test_accuracy( model, test_data, device=device):
  test_loader = DataLoader(test_data, batch_size=200, shuffle=False)
  model.eval()
  correct = 0
  n_data = len(test_data)
  with torch.no_grad():
    for imgs, labels in test_loader:
      imgs = imgs.to(device)
      labels = labels.to(device)
      outputs = model(imgs)
      predicted = outputs.argmax(dim=1)
      correct += (predicted == labels).sum().item()
  return correct / n_data

In [16]:
# one experiment
#return pooling_index new train_index
def run_once(train_index, pooling_index, test_data=test_data, acq_size=10):
  print(f"curr size of train_data {len(train_index)}, curr size of pooling data {len(pooling_index)}  ")
  trainData = NewDataset(index=train_index)
  # vaildData = NewDataset(index=valid_index)
  poolingData =  NewDataset(index=pooling_index)
  model = train_model(trainData=trainData)
  new_trainData_index, new_pool_index = random_choose(pooling_index, acq_size=acq_size)
  train_index.extend(new_trainData_index)
  # print(len(set(new_trainData_index)))
  test_ac = test_accuracy(model, test_data)
  print(f"test accuracy is {test_ac}")
  return train_index, new_pool_index, test_ac


In [18]:
# number of experiement
pooling_index_temp = pooling_index.copy()
train_index_temp = train_index.copy()
print(len(set(pooling_index_temp)))
n_experiement = 50
test_accuracy_lst = []
for i in range(n_experiement):
    train_index_temp, pooling_index_temp, test_ac= run_once(train_index_temp, pooling_index_temp, acq_size=5)
    print(len(set(train_index_temp)))
    test_accuracy_lst.append(test_ac)



3000
curr size of train_data 20, curr size of pooling data 9000  


100%|██████████| 50/50 [00:01<00:00, 25.40it/s]


test accuracy is 0.5544
25
curr size of train_data 25, curr size of pooling data 8985  


100%|██████████| 50/50 [00:02<00:00, 17.42it/s]


test accuracy is 0.565
30
curr size of train_data 30, curr size of pooling data 8970  


...

test accuracy is 0.867
260
curr size of train_data 260, curr size of pooling data 8280  


100%|██████████| 50/50 [00:24<00:00,  2.03it/s]


test accuracy is 0.8742
264
curr size of train_data 265, curr size of pooling data 8268  


100%|██████████| 50/50 [00:24<00:00,  2.06it/s]


test accuracy is 0.885
269


In [ ]:
def save_accuracy(file_name, accuracy_lst):
  path = '/content/drive/MyDrive/UDL_DATA/'
  path = path + file_name
  with open(path, 'w') as f:
      for item in accuracy_lst:
          f.write(f"{item}\n")



In [ ]:
save_accuracy("Novel_randomChoose_iter_batch_size3.txt",test_accuracy_lst )